
# Notebook 13 — F01: FINAL model trained on ALL 707 images (train + validation)
Solar Filament Segmentation Challenge 2026 — Kaggle edition

**Why.** Four submissions (B03, B04, the ensemble, the confidence filter) all scored **0.33**. Architecture, ensembling and post-processing have plateaued; what is left that is
*likely* (not certain) to help is more data: the validation images (139) were never used for training. This notebook trains the B04 configuration on **all 707 labelled images**.

**Important differences from B03/B04 — read before interpreting anything:**
* All images are in training, so there is **no held-out validation**. The validation-style numbers printed at the end are measured on images the model has seen and are
  **a sanity check only (expect values well above 0.44), NOT a performance estimate.** The honest evidence for this model is that it uses the identical configuration
  that scored 0.433-0.444 on validation / 0.33 on the leaderboard.
* The final model is simply the **last epoch** (BN re-estimated). This is safe because in B03/B04 the last epoch matched the best epoch.
* The notebook also saves `final_weights_fp16.pth` (half-precision, ~65 MB for ResNet-50) for the GitHub repository.

**Unchanged:** BCE+Dice loss, Adam (wd 1e-5), 1024 px, batch 4, 50 epochs, AMP, flips-only augmentation, date-grouped split (seed 42), BN re-estimation before every validation,
the inference recipe (flip TTA, threshold 0.5, min 400 px, closing 10), official-protocol validation.

**Setup (Kaggle right panel):** Add Input -> the competition ONLY. **Session options -> Internet ON** (needed once to download the ImageNet weights). GPU on. *Save & Run All* (~2.7 h for ResNet-50, ~1.8 h for ResNet-34).
If a session stops: add this notebook's previous version output as an Input and run again - it resumes.

**Reference to beat on validation (official protocol, same recipe):** B03 **0.4333** (Baseline-01 0.3719, B02a 0.3836; leaderboard 0.28 / 0.29).

In [ ]:
# ============================================================
# CONFIGURATION
# ============================================================
KAGGLE_DATA_CANDIDATES = [
    "/kaggle/input/competitions/filament-segmentation-2026",
    "/kaggle/input/filament-segmentation-2026",
]

ENCODER = "resnet50"                    # "resnet34" | "resnet50" | "resnet101"
TRAIN_SEED_OFFSET = 0                   # 0, 1, 2 ... (split is NOT affected)
assert ENCODER in ("resnet34", "resnet50", "resnet101")
EXPERIMENT = f"F01_{ENCODER}_s{TRAIN_SEED_OFFSET}_full"
RUN_MODE = "FULL"                       # names kept for the shared helper cells
DEBUG_MODE = False
OUTPUT_DIR = "/kaggle/working"
SEED = 42
RESUME = True

CFG = {
    "experiment_id": EXPERIMENT,
    "train_resolution": 1024,
    "batch_size": 4,                    # if the preflight reports CUDA out-of-memory, set 2 here and re-run
    "epochs": 50,
    "lr": 1e-3,                         # decoder learning rate
    "lr_encoder": 1e-4,                 # pretrained encoder learning rate
    "weight_decay": 1e-5,
    "model": f"{ENCODER}-UNet(ImageNet)",
    "encoder": ENCODER,
    "train_seed_offset": TRAIN_SEED_OFFSET,
    "loss": "BCE+Dice",
    "prob_threshold": 0.5,
    "min_instance_px": 0,
    "debug_n_train": 24,
    "debug_n_val": 8,
    "use_amp": True,
    "num_workers": None,
    "pq_eval_every": 0,                 # no monitoring PQ: validation images are part of the training set here
    "train_on_all": True,               # FINAL model: train on train + validation images
    "snapshot_every": 10,
    "ckpt_every": 2,
    "fsync_checkpoints": False,
    "multi_record": True,               # random annotator record per training sample (B02a)
    "cosine_lr": True,                  # cosine decay to 1% of the base learning rates
    "eta_min": 1e-5,                    # decoder end LR (encoder end LR scales proportionally)
    "bn_recal_batches": 60,
    "recipe": {"tta": True, "threshold": 0.5, "min_px": 400, "closing_r": 10},
}
print(f"EXPERIMENT={EXPERIMENT} | pretrained {ENCODER} encoder | seed offset {TRAIN_SEED_OFFSET} | multi_record={CFG['multi_record']} | cosine_lr={CFG['cosine_lr']} | epochs={CFG['epochs']}")

In [ ]:
import copy
import os, sys, json, random, re, warnings, math, shutil, time, hashlib, platform, gc, datetime
from pathlib import Path
from collections import defaultdict, Counter

try:
    import pycocotools
except ImportError:
    !pip -q install pycocotools

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import cv2
from PIL import Image
from tqdm.auto import tqdm
from pycocotools import mask as mask_utils

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

warnings.filterwarnings("ignore")

def set_seed(seed=SEED):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)

set_seed()
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

# ---- GPU is REQUIRED. No silent CPU fallback. ----
if not torch.cuda.is_available():
    raise RuntimeError(
        "No CUDA GPU is available. This notebook will NOT fall back to CPU.\n"
        "Fix: in the Kaggle right-hand panel open Session options -> Accelerator -> choose GPU T4 x2 or P100, "
        "then run again.")
DEVICE = torch.device("cuda:0")
N_GPUS_VISIBLE = torch.cuda.device_count()
print(f"Python {sys.version.split()[0]} | torch {torch.__version__} | device {DEVICE}")
print(f"GPU: {torch.cuda.get_device_name(0)} | visible GPUs: {N_GPUS_VISIBLE} "
      f"| this notebook uses ONLY cuda:0 (no DataParallel)")

if CFG["num_workers"] is None:
    CFG["num_workers"] = min(4, os.cpu_count() or 2)

# ---- output location (Kaggle working dir) ----
OUT_ROOT = Path(OUTPUT_DIR); OUT_ROOT.mkdir(parents=True, exist_ok=True)
OUT = OUT_ROOT / EXPERIMENT
OUT.mkdir(parents=True, exist_ok=True)
IN_KAGGLE = Path("/kaggle/input").exists()
print("Running on Kaggle:", IN_KAGGLE, "| Outputs:", OUT)
print("NOTE: /kaggle/working is kept only when you Save Version (Save & Run All); lost when the session ends otherwise.")

# ---- bring a previous version's checkpoints into /kaggle/working (resume across sessions) ----
RESUME_FILES = ["latest_checkpoint.pth", "latest_checkpoint_prev.pth", "baseline_best.pth",
                "baseline_best_meta.json", "training_history.csv", "split_metadata.csv"]

def import_previous_output():
    if (OUT / "latest_checkpoint.pth").exists() or not IN_KAGGLE:
        return
    hits = [p for p in Path("/kaggle/input").rglob("latest_checkpoint*.pth") if p.parent.name == OUT.name]
    if not hits:
        return
    src = max(hits, key=lambda p: p.stat().st_mtime).parent
    print(f">>> Found a previous run's checkpoints in {src} — copying them into {OUT} for resume")
    for f in RESUME_FILES + [p.name for p in src.glob("checkpoint_epoch_*.pth")]:
        if (src / f).exists() and not (OUT / f).exists():
            shutil.copyfile(src / f, OUT / f)

import_previous_output()
print("DataLoader workers:", CFG["num_workers"])

In [ ]:
# ============================================================
# DATA DISCOVERY on Kaggle (/kaggle/input) — same discovery logic as Notebook 03
# ============================================================

def acquire_data_root():
    for c in KAGGLE_DATA_CANDIDATES:
        if Path(c).exists():
            return Path(c)
    base = Path("/kaggle/input")
    if base.exists() and any(base.rglob("train_images")):
        print("WARNING: standard path not found; using /kaggle/input and auto-discovery.")
        return base
    raise FileNotFoundError(
        "Competition data not found under /kaggle/input.\n"
        "Fix: right-hand panel -> Add Input -> search 'filament-segmentation-2026' -> Add.")

DATA_ROOT = acquire_data_root()
print("DATA_ROOT:", DATA_ROOT)

IMG_EXTS = {".png", ".jpg", ".jpeg", ".tif", ".tiff"}

def discover_dataset(root):
    """Find train/test image dirs and the COCO annotation JSON. Fail loudly."""
    img_dirs = []
    for d in sorted({p.parent for p in root.rglob("*") if p.suffix.lower() in IMG_EXTS}):
        n = sum(1 for p in d.iterdir() if p.suffix.lower() in IMG_EXTS)
        img_dirs.append((d, n))
        print(f"  image dir: {d.relative_to(root)}  ({n} images)")
    train_cands = [d for d, _ in img_dirs
                   if "train" in str(d).lower() and "test" not in str(d).lower()]
    test_cands = [d for d, _ in img_dirs
                  if "test" in str(d).lower() and "train" not in str(d).lower()]
    if len(train_cands) != 1 or len(test_cands) != 1:
        raise RuntimeError(
            f"Ambiguous dataset layout. train candidates={train_cands}, "
            f"test candidates={test_cands}. Inspect the listing above.")
    coco = None
    for p in root.rglob("*.json"):
        try:
            with open(p) as f:
                d = json.load(f)
            if isinstance(d, dict) and "images" in d and "annotations" in d:
                coco = p; break
        except Exception:
            pass
    if coco is None:
        raise RuntimeError(f"No COCO-style annotation JSON found under {root}")
    return train_cands[0], test_cands[0], coco

TRAIN_DIR, TEST_DIR, ANN_PATH = discover_dataset(DATA_ROOT)

train_paths = sorted(p for p in TRAIN_DIR.iterdir() if p.suffix.lower() in IMG_EXTS)
test_paths  = sorted(p for p in TEST_DIR.iterdir()  if p.suffix.lower() in IMG_EXTS)
name_to_path = {p.name: p for p in train_paths}

print(f"\ntrain images: {len(train_paths)} | test images: {len(test_paths)}")
print(f"annotations: {ANN_PATH.name}")
assert len(train_paths) > 0 and len(test_paths) > 0

In [ ]:
# ============================================================
# ANNOTATIONS + BASELINE GT REPRESENTATION  (unchanged from Notebook 03)
# Baseline target = FIRST annotation record per file (sorted by id).
# ============================================================

def load_annotations(path):
    with open(path) as f:
        coco = json.load(f)
    images_info = {im["id"]: im for im in coco["images"]}
    anns_by_image = defaultdict(list)
    for a in coco["annotations"]:
        anns_by_image[a["image_id"]].append(a)
    return coco, images_info, anns_by_image

COCO, images_info, anns_by_image = load_annotations(ANN_PATH)
print(f"JSON records: {len(images_info)} | annotations: {len(COCO['annotations'])}")

file_to_records = defaultdict(list)
for img_id, info in images_info.items():
    fname = Path(info["file_name"]).name
    if fname in name_to_path:
        file_to_records[fname].append(img_id)

missing = len(images_info) - sum(len(v) for v in file_to_records.values())
if missing:
    raise RuntimeError(f"{missing} JSON records reference files not on disk")

file_to_primary = {f: sorted(ids)[0] for f, ids in file_to_records.items()}
primary_ids = sorted(file_to_primary.values())
print(f"physical files: {len(file_to_primary)} | primary records: {len(primary_ids)}")

def ann_to_mask(ann, h, w):
    m = np.zeros((h, w), np.uint8)
    for poly in ann["segmentation"]:
        pts = np.asarray(poly, np.float64).reshape(-1, 2)
        cv2.fillPoly(m, [np.round(pts).astype(np.int32)], 1)
    return m.astype(bool)

def build_gt_masks(img_id):
    """Full-resolution instance masks for one record."""
    info = images_info[img_id]
    masks = [ann_to_mask(a, info["height"], info["width"])
             for a in anns_by_image[img_id]]
    masks = [m for m in masks if m.any()]
    if not masks:
        raise ValueError(f"Record {img_id} has no non-empty GT masks")
    return masks

def load_image(path):
    img = np.array(Image.open(path))
    if img.ndim == 3:
        img = img[..., :3].mean(axis=2)
    return img.astype(np.float32)

In [ ]:
# ============================================================
# OFFICIAL EVALUATOR REPRODUCTION
# Many-to-many matching: EVERY (pred, gt) pair with IoU > 0.5 (STRICT >)
# is a TP and contributes its IoU to the numerator.
# FP = pred with zero qualifying matches; FN = gt with zero.
# PQ = sum(TP IoUs) / (TP + 0.5 FP + 0.5 FN)
# Diagnostics (one-to-many / many-to-one) use overlap > 0.
#
# Only change vs Notebook 03: the pairwise IoU/Dice loop skips pairs whose
# bounding boxes do not intersect and only intersects the bbox overlap region.
# Results are mathematically identical (verified against the original
# implementation in the equivalence test below); it is just much faster.
# ============================================================

def _bbox(m):
    rows = np.flatnonzero(m.any(axis=1)); cols = np.flatnonzero(m.any(axis=0))
    if len(rows) == 0:
        return (0, 0, 0, 0)
    return (rows[0], rows[-1] + 1, cols[0], cols[-1] + 1)

def iou_and_dice_matrix(pred_masks, gt_masks):
    n_p, n_g = len(pred_masks), len(gt_masks)
    iou = np.zeros((n_p, n_g)); dce = np.zeros((n_p, n_g))
    pb = [_bbox(p) for p in pred_masks]; gb = [_bbox(g) for g in gt_masks]
    pa = [int(p.sum()) for p in pred_masks]; ga = [int(g.sum()) for g in gt_masks]
    for i, p in enumerate(pred_masks):
        py0, py1, px0, px1 = pb[i]
        for j, g in enumerate(gt_masks):
            gy0, gy1, gx0, gx1 = gb[j]
            y0, y1 = max(py0, gy0), min(py1, gy1)
            x0, x1 = max(px0, gx0), min(px1, gx1)
            if y0 >= y1 or x0 >= x1:
                continue
            inter = int(np.logical_and(p[y0:y1, x0:x1], g[y0:y1, x0:x1]).sum())
            if inter == 0:
                continue
            iou[i, j] = inter / (pa[i] + ga[j] - inter)
            dce[i, j] = 2 * inter / (pa[i] + ga[j])
    return iou, dce

def compute_official_pq(gt_masks, pred_masks):
    if len(gt_masks) == 0 and len(pred_masks) == 0:
        return {"PQ": 1.0, "TP": 0, "FP": 0, "FN": 0,
                "tp_ious": [], "tp_dices": [],
                "one_to_many": 0, "many_to_one": 0}
    iou, dce = iou_and_dice_matrix(pred_masks, gt_masks)
    hit = iou > 0.5                                   # STRICT >
    tp_ious = iou[hit].tolist()
    tp_dices = dce[hit].tolist()
    tp = len(tp_ious)
    fp = int((hit.sum(axis=1) == 0).sum()) if len(pred_masks) else 0
    fn = int((hit.sum(axis=0) == 0).sum()) if len(gt_masks) else 0
    denom = tp + 0.5 * fp + 0.5 * fn
    overlap = iou > 0
    one_to_many = int((overlap.sum(axis=0) > 1).sum())   # GT hit by multiple preds
    many_to_one = int((overlap.sum(axis=1) > 1).sum())   # pred hitting multiple GTs
    return {"PQ": (sum(tp_ious) / denom) if denom > 0 else 0.0,
            "TP": tp, "FP": fp, "FN": fn,
            "tp_ious": tp_ious, "tp_dices": tp_dices,
            "one_to_many": one_to_many, "many_to_one": many_to_one}

# ---------------- MANDATORY UNIT TESTS (same 7 as Notebook 03) ----------------
def _box(y0, y1, x0, x1, hw=64):
    m = np.zeros((hw, hw), bool); m[y0:y1, x0:x1] = True; return m

def run_metric_tests():
    results = []
    g = _box(10, 20, 10, 30)                       # 10x20 = 200 px

    r = compute_official_pq([g], [g.copy()])
    results.append(("T1 perfect -> PQ=1", abs(r["PQ"] - 1.0) < 1e-9))

    half = _box(10, 20, 10, 20)                    # IoU = 0.5 exactly
    r = compute_official_pq([g], [half])
    results.append(("T2 IoU==0.5 not TP", r["TP"] == 0 and r["FP"] == 1 and r["FN"] == 1))

    most = _box(10, 20, 10, 24)                    # IoU = 0.7
    r = compute_official_pq([g], [most])
    results.append(("T3 IoU>0.5 is TP", r["TP"] == 1 and r["FP"] == 0 and r["FN"] == 0))

    p1 = _box(10, 20, 10, 27); p2 = _box(10, 20, 12, 30)
    r = compute_official_pq([g], [p1, p2])
    results.append(("T4 1 GT + 2 preds -> TP=2 FP=0 FN=0",
                    r["TP"] == 2 and r["FP"] == 0 and r["FN"] == 0))

    g1 = _box(10, 20, 10, 30); g2 = _box(10, 20, 8, 32); pred = _box(10, 20, 9, 31)
    r = compute_official_pq([g1, g2], [pred])
    results.append(("T5 2 GT + 1 pred -> TP=2 FP=0 FN=0",
                    r["TP"] == 2 and r["FP"] == 0 and r["FN"] == 0))

    r = compute_official_pq([g], [])
    results.append(("T6 no preds -> FN=1, PQ=0", r["FN"] == 1 and r["PQ"] == 0.0))

    r = compute_official_pq([], [g])
    results.append(("T7 no GT -> FP=1, PQ=0", r["FP"] == 1 and r["PQ"] == 0.0))

    all_ok = True
    for name, ok in results:
        print(f"  {'PASS' if ok else 'FAIL'}  {name}")
        all_ok &= ok
    if not all_ok:
        raise AssertionError("Metric unit tests FAILED — do not proceed")
    print("All 7 metric tests passed.")

run_metric_tests()

# ---------------- EQUIVALENCE TEST: fast matrix == original Notebook-03 matrix ----------------
def _iou_and_dice_matrix_reference(pred_masks, gt_masks):
    """Verbatim original implementation from Notebook 03 (slow, full-frame)."""
    n_p, n_g = len(pred_masks), len(gt_masks)
    iou = np.zeros((n_p, n_g)); dce = np.zeros((n_p, n_g))
    for i, p in enumerate(pred_masks):
        ps = p.sum()
        for j, g in enumerate(gt_masks):
            inter = np.logical_and(p, g).sum()
            if inter == 0:
                continue
            gs = g.sum()
            iou[i, j] = inter / (ps + gs - inter)
            dce[i, j] = 2 * inter / (ps + gs)
    return iou, dce

def run_equivalence_test(n_trials=40, hw=96):
    rng = np.random.RandomState(0)
    def rand_mask():
        m = np.zeros((hw, hw), np.uint8)
        for _ in range(rng.randint(1, 4)):
            y, x = rng.randint(0, hw - 8, 2); h, w = rng.randint(2, 30, 2)
            m[y:y + h, x:x + w] = 1
        return m.astype(bool)
    for _ in range(n_trials):
        P = [rand_mask() for _ in range(rng.randint(0, 6))]
        G = [rand_mask() for _ in range(rng.randint(0, 6))]
        a = iou_and_dice_matrix(P, G); b = _iou_and_dice_matrix_reference(P, G)
        assert np.allclose(a[0], b[0]) and np.allclose(a[1], b[1]), "fast IoU/Dice differs from original"
    print(f"Equivalence test passed: fast evaluator == original on {n_trials} random cases.")

run_equivalence_test()

In [ ]:
from pycocotools import mask as mask_utils

# ============================================================
# RLE (pycocotools, Fortran order, 2048x2048)  — kept for the later submission notebook
# ============================================================

def encode_rle(mask):
    rle = mask_utils.encode(np.asfortranarray(mask.astype(np.uint8)))
    rle["counts"] = rle["counts"].decode("utf-8")
    return rle

def decode_rle(rle):
    r = dict(rle)
    if isinstance(r["counts"], str):
        r["counts"] = r["counts"].encode("utf-8")
    return mask_utils.decode(r).astype(bool)

_t = np.zeros((2048, 2048), bool)
_t[100:130, 400:900] = True; _t[1500:1510, 30:80] = True
_rt = decode_rle(encode_rle(_t))
assert _rt.shape == (2048, 2048) and (_rt == _t).all(), "RLE round-trip mismatch"
assert decode_rle(encode_rle(np.zeros((2048, 2048), bool))).sum() == 0
print("RLE round-trip checks passed (2048x2048, incl. empty mask)")

In [ ]:
# ============================================================
# SPLIT — date-grouped 80/20 (Notebook 02 A3). Same function, same seed as Notebook 03.
# ============================================================

FN_RE = re.compile(r"^(\d{4})(\d{2})(\d{2})\d{6}([A-Za-z])h\.", re.I)

def create_split(primary_ids, val_frac=0.2, seed=SEED):
    rows = []
    for img_id in primary_ids:
        fname = Path(images_info[img_id]["file_name"]).name
        m = FN_RE.match(fname)
        if not m:
            raise ValueError(f"Unparseable filename: {fname}")
        rows.append({"img_id": img_id, "file": fname,
                     "date": f"{m[1]}-{m[2]}-{m[3]}", "station": m[4].upper()})
    df = pd.DataFrame(rows)
    dates = sorted(df["date"].unique())
    rng = random.Random(seed); rng.shuffle(dates)
    n_val_dates = int(round(len(dates) * val_frac))
    val_dates = set(dates[:n_val_dates])
    df["split"] = df["date"].map(lambda d: "val" if d in val_dates else "train")
    return df

split_df = create_split(primary_ids)
train_ids = split_df.loc[split_df.split == "train", "img_id"].tolist()
val_ids   = split_df.loc[split_df.split == "val", "img_id"].tolist()

if DEBUG_MODE:                       # smoke-test shortcut — must be OFF for Baseline-01
    train_ids = train_ids[:CFG["debug_n_train"]]
    val_ids   = val_ids[:CFG["debug_n_val"]]

if CFG["train_on_all"]:
    train_ids = list(train_ids) + list(val_ids)     # FINAL MODEL: all images are used for training; val_ids kept only for a sanity check

SPLIT_HASH = hashlib.sha1(",".join(sorted(
    split_df.loc[split_df.split == "val", "file"])).encode()).hexdigest()[:12]

split_df.to_csv(OUT / "split_metadata.csv", index=False)
print(f"train: {len(train_ids)} | val: {len(val_ids)} (date-grouped) | split hash: {SPLIT_HASH}")
assert not set(split_df[split_df.split == "train"].date) & \
           set(split_df[split_df.split == "val"].date)

In [ ]:
# ============================================================
# DATASET — identical preprocessing/augmentation to Baseline-01.
# NEW (B02a only): in TRAINING, each sample draws a random annotation record of the image (among records that contain
# filaments). Validation always uses the primary record, exactly as before.
# ============================================================
def usable_records(fname):
    return [r for r in sorted(file_to_records[fname]) if len(anns_by_image.get(r, [])) > 0]

class FilamentDataset(Dataset):
    def __init__(self, record_ids, resolution, augment=False, multi_record=False):
        self.ids = record_ids
        self.res = resolution
        self.augment = augment
        self.multi = multi_record

    def __len__(self):
        return len(self.ids)

    def __getitem__(self, idx):
        img_id = self.ids[idx]
        info = images_info[img_id]
        fname = Path(info["file_name"]).name
        rec = img_id
        if self.augment and self.multi:
            rec = random.choice(usable_records(fname) or [img_id])
        img = load_image(name_to_path[fname])
        h, w = img.shape

        gt = np.zeros((h, w), bool)
        for a in anns_by_image[rec]:
            gt |= ann_to_mask(a, h, w)

        img = cv2.resize(img, (self.res, self.res), interpolation=cv2.INTER_AREA)
        gt = cv2.resize(gt.astype(np.uint8), (self.res, self.res), interpolation=cv2.INTER_NEAREST).astype(bool)

        lo, hi = np.percentile(img, [1, 99])
        img = np.clip((img - lo) / max(hi - lo, 1e-6), 0, 1).astype(np.float32)

        if self.augment:
            if random.random() < 0.5:
                img = np.fliplr(img).copy(); gt = np.fliplr(gt).copy()
            if random.random() < 0.5:
                img = np.flipud(img).copy(); gt = np.flipud(gt).copy()

        return {"image": torch.from_numpy(img)[None],
                "mask": torch.from_numpy(gt.astype(np.float32))[None],
                "img_id": img_id}

def _worker_init(_):
    cv2.setNumThreads(0)

train_ds = FilamentDataset(train_ids, CFG["train_resolution"], augment=True, multi_record=CFG["multi_record"])
val_ds   = FilamentDataset(val_ids, CFG["train_resolution"], augment=False)

train_gen = torch.Generator(); train_gen.manual_seed(SEED)
train_loader = DataLoader(train_ds, batch_size=CFG["batch_size"], shuffle=True,
                          num_workers=CFG["num_workers"], pin_memory=True, drop_last=True,
                          generator=train_gen, worker_init_fn=_worker_init)
val_loader = DataLoader(val_ds, batch_size=1, shuffle=False, num_workers=CFG["num_workers"],
                        pin_memory=True, worker_init_fn=_worker_init)

n_multi = sum(len(usable_records(Path(images_info[i]["file_name"]).name)) > 1 for i in train_ids)
print(f"train batches/epoch: {len(train_loader)} | val images: {len(val_ds)} | "
      f"training files with >=2 usable annotator records: {n_multi} of {len(train_ids)} "
      f"| random-record sampling: {CFG['multi_record']}")

In [ ]:
import torchvision
import torch.nn.functional as F

# ============================================================
# MODEL — plain U-Net (base 32), UNCHANGED.  Loss — BCE + soft Dice, UNCHANGED
# (logits are cast to float32 inside the loss so AMP cannot affect the loss math).
# ============================================================

class ConvBlock(nn.Module):
    def __init__(self, cin, cout):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(cin, cout, 3, padding=1), nn.BatchNorm2d(cout), nn.ReLU(inplace=True),
            nn.Conv2d(cout, cout, 3, padding=1), nn.BatchNorm2d(cout), nn.ReLU(inplace=True))
    def forward(self, x): return self.net(x)

class UNet(nn.Module):
    def __init__(self, cin=1, base=32):
        super().__init__()
        self.e1, self.e2 = ConvBlock(cin, base), ConvBlock(base, base*2)
        self.e3, self.e4 = ConvBlock(base*2, base*4), ConvBlock(base*4, base*8)
        self.pool = nn.MaxPool2d(2)
        self.mid = ConvBlock(base*8, base*16)
        self.u4 = nn.ConvTranspose2d(base*16, base*8, 2, 2); self.d4 = ConvBlock(base*16, base*8)
        self.u3 = nn.ConvTranspose2d(base*8, base*4, 2, 2);  self.d3 = ConvBlock(base*8, base*4)
        self.u2 = nn.ConvTranspose2d(base*4, base*2, 2, 2);  self.d2 = ConvBlock(base*4, base*2)
        self.u1 = nn.ConvTranspose2d(base*2, base, 2, 2);    self.d1 = ConvBlock(base*2, base)
        self.out = nn.Conv2d(base, 1, 1)
    def forward(self, x):
        e1 = self.e1(x); e2 = self.e2(self.pool(e1))
        e3 = self.e3(self.pool(e2)); e4 = self.e4(self.pool(e3))
        m = self.mid(self.pool(e4))
        d = self.d4(torch.cat([self.u4(m), e4], 1))
        d = self.d3(torch.cat([self.u3(d), e3], 1))
        d = self.d2(torch.cat([self.u2(d), e2], 1))
        d = self.d1(torch.cat([self.u1(d), e1], 1))
        return self.out(d)

ENCODER_SPECS = {
    "resnet34":  {"ctor": torchvision.models.resnet34,  "weights": torchvision.models.ResNet34_Weights.IMAGENET1K_V1,  "ch": [64, 64, 128, 256, 512]},
    "resnet50":  {"ctor": torchvision.models.resnet50,  "weights": torchvision.models.ResNet50_Weights.IMAGENET1K_V1,  "ch": [64, 256, 512, 1024, 2048]},
    "resnet101": {"ctor": torchvision.models.resnet101, "weights": torchvision.models.ResNet101_Weights.IMAGENET1K_V1, "ch": [64, 256, 512, 1024, 2048]},
}

class ResNetUNet(nn.Module):
    """U-Net with a ResNet-34 encoder (torchvision). Input: 1-channel image in [0,1]; the model repeats it to 3 channels and
    applies ImageNet mean/std normalisation internally, so the data pipeline stays unchanged."""
    def __init__(self, pretrained=False):
        super().__init__()
        spec = ENCODER_SPECS[CFG["encoder"]]; c = spec["ch"]
        weights = spec["weights"] if pretrained else None
        try:
            r = spec["ctor"](weights=weights)
        except Exception as e:
            raise RuntimeError("Could not download the ImageNet encoder weights. Turn Internet ON "
                               "(Session options -> Internet) and run again. Original error: " + repr(e))
        self.stem = nn.Sequential(r.conv1, r.bn1, r.relu)                 # stride 2,  64 ch
        self.pool = r.maxpool
        self.l1, self.l2, self.l3, self.l4 = r.layer1, r.layer2, r.layer3, r.layer4   # /4:64  /8:128  /16:256  /32:512
        self.register_buffer("mean", torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1))
        self.register_buffer("std", torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1))
        self.d4 = ConvBlock(c[4] + c[3], 256)
        self.d3 = ConvBlock(256 + c[2], 128)
        self.d2 = ConvBlock(128 + c[1], 64)
        self.d1 = ConvBlock(64 + c[0], 64)
        self.d0 = ConvBlock(64, 32)
        self.out = nn.Conv2d(32, 1, 1)

    def encoder_parameters(self):
        for mod in (self.stem, self.l1, self.l2, self.l3, self.l4):
            yield from mod.parameters()

    def decoder_parameters(self):
        for mod in (self.d4, self.d3, self.d2, self.d1, self.d0, self.out):
            yield from mod.parameters()

    def forward(self, x):
        x3 = (x.repeat(1, 3, 1, 1) - self.mean) / self.std
        s0 = self.stem(x3)                      # /2
        s1 = self.l1(self.pool(s0))             # /4
        s2 = self.l2(s1)                        # /8
        s3 = self.l3(s2)                        # /16
        s4 = self.l4(s3)                        # /32
        up = lambda t, ref: F.interpolate(t, size=ref.shape[-2:], mode="bilinear", align_corners=False)
        d = self.d4(torch.cat([up(s4, s3), s3], 1))
        d = self.d3(torch.cat([up(d, s2), s2], 1))
        d = self.d2(torch.cat([up(d, s1), s1], 1))
        d = self.d1(torch.cat([up(d, s0), s0], 1))
        d = self.d0(F.interpolate(d, size=x.shape[-2:], mode="bilinear", align_corners=False))
        return self.out(d)

def build_model(pretrained=False):
    return ResNetUNet(pretrained=pretrained).to(DEVICE)

def build_optimizer(model):
    groups = [{"params": list(model.encoder_parameters()), "lr": CFG["lr_encoder"], "base_lr": CFG["lr_encoder"]},
              {"params": list(model.decoder_parameters()), "lr": CFG["lr"], "base_lr": CFG["lr"]}]
    return torch.optim.Adam(groups, weight_decay=CFG["weight_decay"])

def bce_dice_loss(logits, target, eps=1.0):
    logits = logits.float()
    bce = nn.functional.binary_cross_entropy_with_logits(logits, target)
    prob = torch.sigmoid(logits)
    inter = (prob * target).sum(dim=(2, 3))
    dice = (2 * inter + eps) / (prob.sum(dim=(2, 3)) + target.sum(dim=(2, 3)) + eps)
    return bce + (1 - dice).mean()

USE_AMP = bool(CFG["use_amp"]) and DEVICE.type == "cuda"

def autocast_ctx():
    return torch.autocast(device_type="cuda", dtype=torch.float16, enabled=USE_AMP)

def build_scaler():
    return torch.amp.GradScaler("cuda", enabled=USE_AMP)

_tmp_model = build_model()
N_PARAMS = sum(p.numel() for p in _tmp_model.parameters())
del _tmp_model
print(f"parameters: {N_PARAMS:,} | AMP: {USE_AMP}")

In [ ]:
# ============================================================
# INFERENCE + VALIDATION UNDER THE OFFICIAL METRIC (per-image, full 2048)
# Inference for PQ is done in FP32 (identical to Notebook 03).
# ============================================================

@torch.no_grad()
def predict_image(model, img_path, resolution):
    """Returns full-2048 probability map."""
    img = load_image(img_path)
    h, w = img.shape
    small = cv2.resize(img, (resolution, resolution), interpolation=cv2.INTER_AREA)
    lo, hi = np.percentile(small, [1, 99])
    small = np.clip((small - lo) / max(hi - lo, 1e-6), 0, 1).astype(np.float32)
    x = torch.from_numpy(small)[None, None].to(DEVICE)
    prob = torch.sigmoid(model(x))[0, 0].cpu().numpy()
    return cv2.resize(prob, (w, h), interpolation=cv2.INTER_LINEAR)

def extract_instances(prob, threshold, min_px=0):
    """Probability map -> list of boolean instance masks via connected components."""
    binary = (prob > threshold).astype(np.uint8)
    n, labels = cv2.connectedComponents(binary, connectivity=8)
    out = []
    for k in range(1, n):
        m = labels == k
        if m.sum() > min_px:
            out.append(m)
    return out

def evaluate_validation(model, ids, threshold, desc="validation"):
    """Official-metric evaluation over ALL images in `ids`."""
    model.eval()
    rows, all_tp_ious, all_tp_dices = [], [], []
    agg = Counter()
    for img_id in tqdm(ids, desc=desc, leave=False):
        fname = Path(images_info[img_id]["file_name"]).name
        gt = build_gt_masks(img_id)
        prob = predict_image(model, name_to_path[fname], CFG["train_resolution"])
        preds = extract_instances(prob, threshold, CFG["min_instance_px"])
        r = compute_official_pq(gt, preds)
        all_tp_ious += r["tp_ious"]; all_tp_dices += r["tp_dices"]
        for k in ["TP", "FP", "FN", "one_to_many", "many_to_one"]:
            agg[k] += r[k]
        rows.append({"img_id": img_id, "file": fname,
                     "n_gt": len(gt), "n_pred": len(preds),
                     "PQ": r["PQ"], "TP": r["TP"], "FP": r["FP"], "FN": r["FN"],
                     "one_to_many": r["one_to_many"], "many_to_one": r["many_to_one"],
                     "mean_tp_iou": float(np.mean(r["tp_ious"])) if r["tp_ious"] else np.nan,
                     "mean_tp_dice": float(np.mean(r["tp_dices"])) if r["tp_dices"] else np.nan})
    per_image = pd.DataFrame(rows)
    denom = agg["TP"] + 0.5 * agg["FP"] + 0.5 * agg["FN"]
    overall = {
        "PQ_pooled": float(sum(all_tp_ious) / denom) if denom else 0.0,
        "PQ_mean_per_image": float(per_image["PQ"].mean()),
        "TP": int(agg["TP"]), "FP": int(agg["FP"]), "FN": int(agg["FN"]),
        "mean_tp_iou": float(np.mean(all_tp_ious)) if all_tp_ious else 0.0,
        "mean_tp_dice": float(np.mean(all_tp_dices)) if all_tp_dices else 0.0,
        "one_to_many": int(agg["one_to_many"]), "many_to_one": int(agg["many_to_one"]),
        "n_images_evaluated": int(len(per_image)), "threshold": float(threshold),
    }
    return per_image, overall, all_tp_ious, all_tp_dices

In [ ]:
# ============================================================
# HELPERS: pooled-PQ summary, BN re-estimation, flip-TTA prediction, post-processing
# ============================================================
def summarize(items):
    tp = sum(r["TP"] for r in items); fp = sum(r["FP"] for r in items); fn = sum(r["FN"] for r in items)
    ious = [x for r in items for x in r["tp_ious"]]
    denom = tp + 0.5 * fp + 0.5 * fn
    return {"PQ_pooled": (sum(ious) / denom) if denom else 0.0,
            "PQ_mean_per_image": float(np.mean([r["PQ"] for r in items])) if items else 0.0,
            "TP": tp, "FP": fp, "FN": fn, "mean_tp_iou": float(np.mean(ious)) if ious else 0.0}


def load_model(path):
    ck = torch.load(path, map_location="cpu", weights_only=True)
    sd = ck["model"] if isinstance(ck, dict) and "model" in ck else ck
    m = build_model(); m.load_state_dict(sd); m.eval()
    return m

def recalibrate_bn(model, n_batches=60):
    """Re-estimate BatchNorm running statistics on ~240 training images (weights untouched)."""
    m = copy.deepcopy(model)
    for mod in m.modules():
        if isinstance(mod, nn.BatchNorm2d):
            mod.reset_running_stats(); mod.momentum = None
    g = torch.Generator(); g.manual_seed(SEED)
    loader = DataLoader(train_ds, batch_size=CFG["batch_size"], shuffle=True, num_workers=CFG["num_workers"],
                        drop_last=True, generator=g, worker_init_fn=_worker_init)
    it = iter(loader); m.train()
    with torch.no_grad():
        for _ in range(n_batches):
            b = next(it)
            with autocast_ctx():
                m(b["image"].to(DEVICE))
    del it; m.eval()
    return m

def postprocess(prob, thr, min_px, close_r):
    binary = (prob > thr).astype(np.uint8)
    if close_r:
        binary = cv2.morphologyEx(binary, cv2.MORPH_CLOSE,
                                  cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (2 * close_r + 1, 2 * close_r + 1)))
    n, labels = cv2.connectedComponents(binary, connectivity=8)
    areas = np.bincount(labels.ravel(), minlength=n)
    return [labels == k for k in range(1, n) if areas[k] > min_px]

@torch.no_grad()
def predict_views(model, img_path, resolution, tta=True):
    """Returns (plain prob, flip-TTA-averaged prob) at model resolution + original image shape."""
    img = load_image(img_path)
    small = cv2.resize(img, (resolution, resolution), interpolation=cv2.INTER_AREA)
    lo, hi = np.percentile(small, [1, 99])
    small = np.clip((small - lo) / max(hi - lo, 1e-6), 0, 1).astype(np.float32)
    x = torch.from_numpy(small)[None, None].to(DEVICE)
    views = [None, [3], [2], [2, 3]] if tta else [None]
    outs = []
    for dims in views:
        xi = x if dims is None else torch.flip(x, dims)
        p = torch.sigmoid(model(xi))
        if dims is not None:
            p = torch.flip(p, dims)
        outs.append(p[0, 0].cpu().numpy())
    return outs[0], np.mean(outs, axis=0), img.shape

def upsample(p, shape):
    h, w = shape
    return cv2.resize(p, (w, h), interpolation=cv2.INTER_LINEAR)

In [ ]:
# ============================================================
# CHECKPOINT UTILITIES (atomic writes; latest + previous + best; resume-safe)
# ============================================================

CKPT_BEST      = OUT / "baseline_best.pth"            # state_dict only (same format as Notebook 03)
CKPT_BEST_META = OUT / "baseline_best_meta.json"
CKPT_LATEST    = OUT / "latest_checkpoint.pth"        # full resumable state
CKPT_PREV      = OUT / "latest_checkpoint_prev.pth"   # previous latest (fallback if latest is corrupt)
HIST_CSV       = OUT / "training_history.csv"

CORE_KEYS = ["train_resolution", "batch_size", "lr", "weight_decay", "model", "loss", "use_amp", "multi_record", "cosine_lr", "eta_min", "lr_encoder", "encoder", "train_seed_offset", "train_on_all"]

def cfg_core():
    return {k: CFG[k] for k in CORE_KEYS} | {"seed": SEED}

def _atomic_replace(tmp, dst):
    if CFG["fsync_checkpoints"]:
        try:
            with open(tmp, "rb") as f:
                os.fsync(f.fileno())
        except Exception:
            pass
    os.replace(tmp, dst)

def atomic_torch_save(obj, path):
    tmp = path.with_name(path.name + ".tmp")
    torch.save(obj, tmp)
    _atomic_replace(tmp, path)

def atomic_write_text(path, text):
    tmp = path.with_name(path.name + ".tmp")
    tmp.write_text(text)
    _atomic_replace(tmp, path)

REQUIRED_CKPT_KEYS = {"epoch", "model", "optimizer", "scaler", "history", "best_val",
                      "best_epoch", "cfg_core", "split_hash", "elapsed_train_s"}

def load_checkpoint_if_valid(path):
    if not path.exists():
        return None
    try:
        ck = torch.load(path, map_location="cpu", weights_only=True)
        if not REQUIRED_CKPT_KEYS.issubset(ck.keys()):
            raise ValueError("missing keys: " + str(REQUIRED_CKPT_KEYS - set(ck.keys())))
        if not isinstance(ck["epoch"], int) or ck["epoch"] < 0:
            raise ValueError("bad epoch")
        return ck
    except Exception as e:
        print(f"  WARNING: checkpoint {path.name} is not usable ({e!r})")
        return None

def find_latest_valid_checkpoint():
    for p in (CKPT_LATEST, CKPT_PREV):
        ck = load_checkpoint_if_valid(p)
        if ck is not None:
            return p, ck
    return None, None

def save_latest_checkpoint(model, optimizer, scaler, epoch, history, best_val, best_epoch,
                           elapsed_train_s, snapshot=False):
    state = {"epoch": int(epoch), "model": model.state_dict(), "optimizer": optimizer.state_dict(),
             "scaler": scaler.state_dict(), "history": history,
             "best_val": float(best_val), "best_epoch": None if best_epoch is None else int(best_epoch),
             "cfg_core": cfg_core(), "split_hash": SPLIT_HASH,
             "elapsed_train_s": float(elapsed_train_s)}
    tmp = CKPT_LATEST.with_name(CKPT_LATEST.name + ".tmp")
    torch.save(state, tmp)
    if CFG["fsync_checkpoints"]:
        try:
            with open(tmp, "rb") as f: os.fsync(f.fileno())
        except Exception:
            pass
    if CKPT_LATEST.exists():
        os.replace(CKPT_LATEST, CKPT_PREV)       # keep one previous valid checkpoint
    os.replace(tmp, CKPT_LATEST)
    if snapshot:
        shutil.copyfile(CKPT_LATEST, OUT / f"checkpoint_epoch_{epoch:03d}.pth")

def save_best(model, epoch, val_loss):
    atomic_torch_save(model.state_dict(), CKPT_BEST)
    atomic_write_text(CKPT_BEST_META, json.dumps(
        {"best_epoch": int(epoch), "best_val_loss": float(val_loss),
         "saved_at": datetime.datetime.now().isoformat(timespec="seconds")}))

def read_best_meta():
    try:
        return json.loads(CKPT_BEST_META.read_text())
    except Exception:
        return None

print("checkpoint dir:", OUT)
_p, _ck = find_latest_valid_checkpoint()
print("latest valid checkpoint:", None if _p is None else f"{_p.name} (completed epoch {_ck['epoch']})")
del _p, _ck

In [ ]:
# ============================================================
# PREFLIGHT
# ============================================================
PREFLIGHT = []
def check(name, ok, detail=""):
    PREFLIGHT.append((name, bool(ok), detail))
    print(f"  [{'PASS' if ok else 'FAIL'}] {name}" + (f"  ->  {detail}" if detail else ""))

print("=" * 78); print(f"PREFLIGHT — {EXPERIMENT}"); print("=" * 78)
check("DEBUG_MODE == False", DEBUG_MODE is False)
check("resolution 1024 / epochs 50", (CFG["train_resolution"], CFG["epochs"]) == (1024, 50),
      f"resolution {CFG['train_resolution']}, epochs {CFG['epochs']}, batch {CFG['batch_size']}")
check("ALL labelled images are used for training", len(train_ids) == len(primary_ids) == len(train_paths) and len(set(train_ids)) == len(train_ids),
      f"training images {len(train_ids)} | primary records {len(primary_ids)} | files on disk {len(train_paths)}")
print("  note: train_on_all=True -> the validation images are INSIDE the training set by design (no held-out validation in this run)")
check("split reproducible (same hash as Baseline-01/02)", SPLIT_HASH == "0cc2d729a5b1", SPLIT_HASH)
props = torch.cuda.get_device_properties(0)
total_gb = props.total_memory / 1024**3
check("CUDA GPU with >= 8 GB", DEVICE.type == "cuda" and total_gb >= 8,
      f"{props.name}, {total_gb:.1f} GB | GPUs visible {N_GPUS_VISIBLE}, used 1")

# ---- pretrained weights must really load (never silently train from scratch) ----
try:
    _m = build_model(pretrained=True)
    n_enc = sum(p.numel() for p in _m.encoder_parameters()); n_dec = sum(p.numel() for p in _m.decoder_parameters())
    check(f"ImageNet {CFG['encoder']} weights downloaded and loaded", True,
          f"encoder {n_enc/1e6:.1f}M params, decoder {n_dec/1e6:.1f}M params, total {(n_enc+n_dec)/1e6:.1f}M")
    del _m
except Exception as e:
    check(f"ImageNet {CFG['encoder']} weights downloaded and loaded", False, str(e)[:300])

# ---- GPU memory/speed probe: one real training step at the configured batch size ----
def probe_gpu():
    torch.cuda.empty_cache(); torch.cuda.reset_peak_memory_stats()
    pm = build_model(False); po = build_optimizer(pm); ps = build_scaler()
    bs, res = CFG["batch_size"], CFG["train_resolution"]
    x = torch.rand(bs, 1, res, res, device=DEVICE); y = (torch.rand(bs, 1, res, res, device=DEVICE) > 0.99).float()
    times = []; pm.train()
    for _ in range(3):
        torch.cuda.synchronize(); t = time.time(); po.zero_grad(set_to_none=True)
        with autocast_ctx():
            out = pm(x)
        loss = bce_dice_loss(out, y); ps.scale(loss).backward(); ps.step(po); ps.update()
        torch.cuda.synchronize(); times.append(time.time() - t)
    peak = torch.cuda.max_memory_allocated()
    del pm, po, ps, x, y, out, loss; gc.collect(); torch.cuda.empty_cache()
    return peak, float(np.median(times[1:]))
try:
    peak_b, sec_it = probe_gpu()
    check("one training step at 1024 px x batch fits in GPU memory (<= 90%)", peak_b <= 0.90 * props.total_memory,
          f"peak {peak_b/1024**3:.1f} GB of {total_gb:.1f} GB")
    print(f"  info: ~{sec_it:.2f} s/iter -> rough {sec_it*len(train_loader)/60:.1f} min/epoch training-only")
except torch.cuda.OutOfMemoryError:
    torch.cuda.empty_cache()
    check("one training step fits in GPU memory", False, f"CUDA OOM at batch {CFG['batch_size']}: set batch_size to 2 in the CONFIG cell and re-run")
check("output dir writable", os.access(OUT, os.W_OK), str(OUT))
bad = [(n, d) for n, ok, d in PREFLIGHT if not ok]
print("=" * 78)
if bad:
    PREFLIGHT_PASSED = False
    raise RuntimeError("PREFLIGHT FAILED:\n" + "\n".join(f" - {n}: {d}" for n, d in bad))
PREFLIGHT_PASSED = True
set_seed(SEED)
print("PREFLIGHT PASSED. Safe to start training.")

In [ ]:
# ============================================================
# TRAINING (resumable). Differences from Baseline-01:
#   * cosine LR (B02b only), set per epoch from the epoch index (resume-safe)
#   * BN statistics re-estimated on a COPY of the model before every validation; validation loss, checkpoint selection and the
#     saved best weights all use that BN-fixed copy. Training itself continues from the raw model.
# ============================================================
assert PREFLIGHT_PASSED, "Run the PREFLIGHT cell first."

def lr_at(epoch):
    if not CFG["cosine_lr"]:
        return CFG["lr"]
    return CFG["eta_min"] + 0.5 * (CFG["lr"] - CFG["eta_min"]) * (1 + math.cos(math.pi * epoch / CFG["epochs"]))

def train_one_epoch(model, optimizer, scaler, loader):
    model.train(); total = 0.0; wait = 0.0
    it = iter(loader)
    for _ in tqdm(range(len(loader)), leave=False, desc="train"):
        tw = time.time(); batch = next(it); wait += time.time() - tw
        img = batch["image"].to(DEVICE, non_blocking=True)
        msk = batch["mask"].to(DEVICE, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        with autocast_ctx():
            logits = model(img)
        loss = bce_dice_loss(logits, msk)
        if not torch.isfinite(loss):
            raise RuntimeError("Non-finite loss — stopping")
        scaler.scale(loss).backward()
        scaler.step(optimizer); scaler.update()
        total += loss.item()
    del it
    return total / len(loader), wait

@torch.no_grad()
def validate_one_epoch(model, loader):
    model.eval(); total = 0.0
    for batch in loader:
        img = batch["image"].to(DEVICE, non_blocking=True)
        msk = batch["mask"].to(DEVICE, non_blocking=True)
        with autocast_ctx():
            logits = model(img)
        total += bce_dice_loss(logits, msk).item()
    return total / len(loader)

RUN_WARNINGS, RESUME_EVENTS = [], []
set_seed(SEED + CFG["train_seed_offset"])
model = build_model(pretrained=True); optimizer = build_optimizer(model); scaler = build_scaler()
history, best_val, best_epoch, start_epoch, elapsed_prior = [], float("inf"), None, 0, 0.0

if RESUME:
    ck_path, ck = find_latest_valid_checkpoint()
    if ck is not None:
        if ck["split_hash"] != SPLIT_HASH:
            raise RuntimeError("Checkpoint was trained on a DIFFERENT split. Refusing to resume.")
        if ck["cfg_core"] != cfg_core():
            raise RuntimeError(f"Checkpoint config differs from current config.\nckpt: {ck['cfg_core']}\nnow:  {cfg_core()}")
        model.load_state_dict(ck["model"]); optimizer.load_state_dict(ck["optimizer"]); scaler.load_state_dict(ck["scaler"])
        history, best_val, best_epoch = ck["history"], ck["best_val"], ck["best_epoch"]
        start_epoch, elapsed_prior = ck["epoch"], ck["elapsed_train_s"]
        RESUME_EVENTS.append({"from_epoch": start_epoch, "checkpoint": ck_path.name})
        print(f">>> RESUMING from {ck_path.name}: {start_epoch} epochs done (best val {best_val:.4f} @ epoch {best_epoch})")
        del ck
    else:
        print(">>> No valid checkpoint found — starting from epoch 1.")

meta = read_best_meta()
if meta is not None and CKPT_BEST.exists():
    if meta["best_val_loss"] < best_val:
        best_val, best_epoch = meta["best_val_loss"], meta["best_epoch"]
elif start_epoch > 0 and best_epoch is not None and not CKPT_BEST.exists():
    RUN_WARNINGS.append("best checkpoint file missing at resume; best-model selection restarted from the resume point.")
    best_val, best_epoch = float("inf"), None
    print("WARNING:", RUN_WARNINGS[-1])

completed_epochs = start_epoch
TRAINING_COMPLETE = (completed_epochs == CFG["epochs"])
if TRAINING_COMPLETE:
    print(f"All {CFG['epochs']} epochs were already completed — skipping training.")

t_session0 = time.time()
try:
    for epoch in range(start_epoch, CFG["epochs"]):
        lr = lr_at(epoch)                                  # decoder LR (reported); encoder LR scales proportionally
        for g in optimizer.param_groups:
            g["lr"] = g["base_lr"] * (lr / CFG["lr"])
        train_gen.manual_seed(SEED + 1000 * CFG["train_seed_offset"] + epoch)
        t0 = time.time()
        tr, data_wait_s = train_one_epoch(model, optimizer, scaler, train_loader)
        t1 = time.time()
        m_eval = recalibrate_bn(model, CFG["bn_recal_batches"])          # BN-fixed COPY; training model untouched
        recal_s = time.time() - t1
        vl = validate_one_epoch(m_eval, val_loader)
        epoch_s = time.time() - t0

        row = {"epoch": epoch + 1, "lr": lr, "train_loss": tr, "val_loss": vl, "epoch_time_s": epoch_s,
               "data_wait_s": data_wait_s, "bn_recal_s": recal_s, "ckpt_save_s": 0.0, "elapsed_train_s": float("nan"),
               "val_PQ_pooled": float("nan"), "val_PQ_mean_per_image": float("nan"), "pq_eval_time_s": float("nan")}
        flag = ""
        t_save0 = time.time()
        if vl < best_val:
            best_val, best_epoch = vl, epoch + 1
            save_best(m_eval, epoch + 1, vl)                              # saves the BN-FIXED weights
            flag = "  * best saved"

        history.append(row); completed_epochs = epoch + 1
        row["elapsed_train_s"] = elapsed_prior + (time.time() - t_session0)
        if completed_epochs % CFG["ckpt_every"] == 0 or completed_epochs == CFG["epochs"]:
            snap = bool(CFG["snapshot_every"] and completed_epochs % CFG["snapshot_every"] == 0)
            save_latest_checkpoint(model, optimizer, scaler, completed_epochs, history, best_val, best_epoch,
                                   row["elapsed_train_s"], snapshot=snap)
            flag += "  [ckpt]"
        row["ckpt_save_s"] = time.time() - t_save0

        pe = CFG["pq_eval_every"]
        if pe and (completed_epochs % pe == 0 or completed_epochs == CFG["epochs"]):
            tq = time.time()
            _, ov, _, _ = evaluate_validation(m_eval, val_ids, CFG["prob_threshold"], desc=f"val PQ @ epoch {completed_epochs}")
            row["val_PQ_pooled"] = ov["PQ_pooled"]; row["val_PQ_mean_per_image"] = ov["PQ_mean_per_image"]
            row["pq_eval_time_s"] = time.time() - tq
            flag += f"  | monitor PQ (primary rec, no filter) {ov['PQ_pooled']:.4f}"
            row["elapsed_train_s"] = elapsed_prior + (time.time() - t_session0)
            save_latest_checkpoint(model, optimizer, scaler, completed_epochs, history, best_val, best_epoch,
                                   row["elapsed_train_s"], snapshot=False)
        del m_eval
        row["elapsed_train_s"] = elapsed_prior + (time.time() - t_session0)
        pd.DataFrame(history).to_csv(HIST_CSV, index=False)
        print(f"epoch {epoch+1:3d}/{CFG['epochs']}  lr {lr:.2e}  train {tr:.4f}  val(BN-fixed) {vl:.4f}  "
              f"(train+val+recal {epoch_s:.0f}s, total {row['elapsed_train_s']/60:.1f} min){flag}")

    TRAINING_COMPLETE = (completed_epochs == CFG["epochs"])

except KeyboardInterrupt:
    TRAINING_COMPLETE = False
    print(f"\nINTERRUPTED after {completed_epochs}/{CFG['epochs']} epochs. TO RESUME: re-run this cell (same session) or add this "
          f"notebook's previous version output as an Input and run all cells; it continues at epoch {completed_epochs + 1}.")
    raise
except torch.cuda.OutOfMemoryError:
    TRAINING_COMPLETE = False
    print("CUDA out of memory (batch size not changed automatically). Completed epochs are safe in the checkpoint.")
    raise

if TRAINING_COMPLETE:
    print(f"\nTRAINING COMPLETE: all {completed_epochs}/{CFG['epochs']} epochs. Best BN-fixed val loss {best_val:.4f} at epoch {best_epoch}.")
    m_last = recalibrate_bn(model, CFG["bn_recal_batches"])
    torch.save(m_last.state_dict(), OUT / "last_epoch_bn_fixed.pth")
    del m_last
else:
    print(f"\nTRAINING NOT COMPLETE: {completed_epochs}/{CFG['epochs']} epochs. Re-run to resume.")

In [ ]:
# ============================================================
# FINAL EVALUATION — official protocol (every annotator record scored separately, pooled), complete validation set
# ============================================================
if not TRAINING_COMPLETE:
    raise RuntimeError(f"Training is NOT complete ({completed_epochs}/{CFG['epochs']} epochs). Re-run the training cell to resume.")

def records_of(img_id):
    return sorted(file_to_records[Path(images_info[img_id]["file_name"]).name])

def official_eval(model, tta, thr, min_px, close_r, desc):
    model.eval(); prim, offi = [], []
    for img_id in tqdm(val_ids, desc=desc, leave=False):
        fn = Path(images_info[img_id]["file_name"]).name
        plain, tta_p, shp = predict_views(model, name_to_path[fn], CFG["train_resolution"], tta=tta)
        preds = postprocess(upsample(tta_p if tta else plain, shp), thr, min_px, close_r)
        prim.append(compute_official_pq(build_gt_masks(img_id), preds))
        for rec in records_of(img_id):
            try:
                gt = build_gt_masks(rec)
            except ValueError:
                continue
            offi.append(compute_official_pq(gt, preds))
    return summarize(prim), summarize(offi)

R = CFG["recipe"]
CANDIDATES = {"best_val_loss (BN-fixed)": OUT / "baseline_best.pth", "last_epoch (BN-fixed)": OUT / "last_epoch_bn_fixed.pth"}
FINAL_ROWS = []
MODELS = {}
for name, path in CANDIDATES.items():
    m = build_model(); m.load_state_dict(torch.load(path, map_location=DEVICE, weights_only=True)); m.eval(); MODELS[name] = m
    p0, o0 = official_eval(m, False, 0.5, 0, 0, f"{name}: plain")
    p1, o1 = official_eval(m, R["tta"], R["threshold"], R["min_px"], R["closing_r"], f"{name}: recipe")
    FINAL_ROWS.append({"model": name, "plain_primary_PQ": p0["PQ_pooled"], "plain_official_PQ": o0["PQ_pooled"],
                       "recipe_primary_PQ": p1["PQ_pooled"], "recipe_official_PQ": o1["PQ_pooled"],
                       "TP": o1["TP"], "FP": o1["FP"], "FN": o1["FN"], "mean_tp_iou": o1["mean_tp_iou"]})
final_df = pd.DataFrame(FINAL_ROWS); final_df.to_csv(OUT / "final_validation.csv", index=False)
print(final_df.round(4).to_string(index=False))
print("\nReferences (official-protocol validation PQ, same recipe): B03 0.4333 | B02a 0.3836 | Baseline-01 0.3719 (leaderboard 0.28-0.29)")

FINAL_NAME = "last_epoch (BN-fixed)"      # no validation-based selection is possible when all images are used for training
FINAL_VAL = float(final_df[final_df.model == FINAL_NAME].iloc[0].recipe_official_PQ)
FINAL_MODEL = MODELS[FINAL_NAME]
torch.save(FINAL_MODEL.state_dict(), OUT / "final_weights.pth")
print(f"\nFinal model for the submission: {FINAL_NAME} (PQ with recipe on images that were IN TRAINING — sanity check only, NOT a validation score {FINAL_VAL:.4f}) -> final_weights.pth")
print("(last epoch; the numbers above are on training images and must not be quoted as performance)")

In [ ]:
# ============================================================
# TRAINING CURVES
# ============================================================
hd = pd.read_csv(HIST_CSV)
fig, axes = plt.subplots(1, 3, figsize=(18, 4.2))
axes[0].plot(hd.epoch, hd.train_loss, label="train loss"); axes[0].plot(hd.epoch, hd.val_loss, label="val loss (BN-fixed)")
axes[0].axvline(best_epoch, c="gray", ls="--", label=f"best epoch ({best_epoch})"); axes[0].legend(); axes[0].set_title("Loss by epoch")
axes[1].plot(hd.epoch, hd.lr); axes[1].set_title("Learning rate"); axes[1].set_yscale("log")
pq = hd.dropna(subset=["val_PQ_pooled"])
axes[2].plot(pq.epoch, pq.val_PQ_pooled, "o-"); axes[2].set_title("Monitor PQ (primary record, no filter, BN-fixed)")
plt.tight_layout(); plt.savefig(OUT / "training_curve.png", dpi=110); plt.show()

In [ ]:
# ============================================================
# TEST SET -> submission_<experiment>.csv   (same format and recipe as the first submission)
# ============================================================
half_sd = {k: (v.half() if v.is_floating_point() else v) for k, v in FINAL_MODEL.state_dict().items()}
torch.save(half_sd, OUT / "final_weights_fp16.pth")
print("saved final_weights_fp16.pth", f"({(OUT / 'final_weights_fp16.pth').stat().st_size / 1e6:.0f} MB) — for the GitHub repository")
rows, per_image_counts = [], []
for path in tqdm(test_paths, desc="test inference"):
    plain, tta_p, shp = predict_views(FINAL_MODEL, path, CFG["train_resolution"], tta=True)
    prob = upsample(tta_p if R["tta"] else plain, shp)
    preds = postprocess(prob, R["threshold"], R["min_px"], R["closing_r"])
    per_image_counts.append(len(preds))
    for k, m_ in enumerate(preds):
        rows.append({"filament_id": f"{path.stem}_{k}", "segmentation_rle": encode_rle(m_)["counts"]})

sub = pd.DataFrame(rows, columns=["filament_id", "segmentation_rle"])
SUB_PATH = Path(OUTPUT_DIR) / f"submission_{EXPERIMENT}.csv"
sub.to_csv(SUB_PATH, index=False); sub.to_csv(OUT / SUB_PATH.name, index=False)
chk = pd.read_csv(SUB_PATH)
assert list(chk.columns) == ["filament_id", "segmentation_rle"] and chk.filament_id.is_unique and chk.segmentation_rle.notna().all()
assert set(chk.filament_id.str.split("_", n=1).str[0]) <= {p.stem for p in test_paths}
n_img_with = int((np.array(per_image_counts) > 0).sum())
print(f"{SUB_PATH.name}: {len(sub)} filaments over {n_img_with}/{len(test_paths)} test images "
      f"(mean {np.mean(per_image_counts):.1f} per image; first submission had 1304 filaments, 7.2 per image)")

In [ ]:
# ============================================================
# SUMMARY — copy this block back to Claude
# ============================================================
hd = pd.read_csv(HIST_CSV)
L = []; add = L.append
add(f"F01 (FULL-DATA) SUMMARY — experiment {EXPERIMENT}")
add("-" * 70)
add(f"epochs completed: {len(hd)}/{CFG['epochs']} | training time {hd.elapsed_train_s.iloc[-1]/3600:.2f} h | resumed: {bool(RESUME_EVENTS)}")
add(f"model: {CFG['model']} | lr enc/dec {CFG['lr_encoder']}/{CFG['lr']} | multi_record={CFG['multi_record']} cosine_lr={CFG['cosine_lr']} | BN re-estimation before every validation: yes")
add(f"best BN-fixed val loss {best_val:.4f} at epoch {best_epoch}")
add("val-loss curve (every 5th epoch): " + ", ".join(f"{int(r.epoch)}:{r.val_loss:.3f}" for r in hd.iloc[4::5].itertuples()))
add("official-protocol validation (model | plain PQ | recipe PQ | TP FP FN):")
for r in final_df.itertuples():
    add(f"   {r.model:<26} {r.plain_official_PQ:.4f} | {r.recipe_official_PQ:.4f} | {r.TP} {r.FP} {r.FN}   (primary-record recipe PQ {r.recipe_primary_PQ:.4f})")
add(f"final model: {FINAL_NAME} | PQ on TRAINING images (sanity only) {FINAL_VAL:.4f}")
add(f"submission file: {SUB_PATH.name} ({len(sub)} filaments, {n_img_with}/{len(test_paths)} images)")
add(f"warnings: {RUN_WARNINGS if RUN_WARNINGS else 'none'}")
add("CAVEAT: all images were used for training -> no held-out validation; submit and compare with the leaderboard 0.33 of B04 (same configuration).")
rep = "\n".join(L); print(rep); (OUT / "baseline02_report.txt").write_text(rep)
print("\nSaved to", OUT, ":", sorted(p.name for p in OUT.iterdir()))